# Exp03 Endogenous-K Publication Table

Exp03 extends the baseline pricing-rule experiment by letting each seller choose a composite action made of a pricing rule and a commitment length `K`. The scan fixes `N=3`, uses `base_K=10`, lets agents choose `K in {10, 30, 60}`, and compares the 3-rule and 4-rule action sets. This notebook summarizes the `mu=0.25` slice into a two-row publication table.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import great_tables as gt

MU_BASELINE = 0.25
N_VALUE = 3

PROJECT_ROOT = Path("../..").resolve()
RESULTS_DIR = PROJECT_ROOT / "data" / "results_exp03"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "robust_exp03_K_action"

EXPERIMENT_DIRS = {
    "3 Rules": RESULTS_DIR / "scan_kchoice_3strats_mu0.25_K10-30-60",
    "4 Rules": RESULTS_DIR / "scan_kchoice_4strats_mu0.25_K10-30-60",
}

sys.path.append(str(PROJECT_ROOT))

from src_ext_K_action.environment import compute_nash_and_monopoly_static

In [2]:
def summarize_experiment(rules_label: str, exp_dir: Path) -> dict:
    cfg = json.load(open(exp_dir / f"Config_N_{N_VALUE}.json"))
    p_nash, p_monopoly = compute_nash_and_monopoly_static(
        cfg["num_sellers"],
        cfg["a_val"],
        cfg["mu"],
        cfg["a0"],
        cfg["c_val"],
    )

    step = (p_monopoly - p_nash) / (cfg["num_grids"] - 3)
    price_grid = np.linspace(p_nash - step, p_monopoly + step, cfg["num_grids"])
    if price_grid[0] < cfg["c_val"]:
        price_grid[0] = cfg["c_val"]

    deltas = []
    lowest_price_means = []
    parquet_files = sorted(
        p for p in (exp_dir / f"N_{N_VALUE}").glob("run_*.parquet")
        if "_qtable" not in p.name
    )

    for parquet_file in parquet_files:
        df = pd.read_parquet(parquet_file, columns=["delta", "price_min"])
        deltas.append(float(df["delta"].mean()))
        lowest_price_means.append(float(df["price_min"].mean()))

    return {
        "Algo. Rules": rules_label,
        "N": N_VALUE,
        "Delta Mean": float(np.mean(deltas)),
        "Delta SD": float(np.std(deltas, ddof=1)),
        "Price Grid Min": float(price_grid[0]),
        "Price Grid Max": float(price_grid[-1]),
        "Lowest Price Mean": float(np.mean(lowest_price_means)),
        "Lowest Price SD": float(np.std(lowest_price_means, ddof=1)),
    }


table_df = pd.DataFrame(
    [summarize_experiment(label, exp_dir) for label, exp_dir in EXPERIMENT_DIRS.items()]
)

table_df[r"$\Delta$"] = table_df.apply(
    lambda row: f"{row['Delta Mean']:.2f}\n({row['Delta SD']:.2f})",
    axis=1,
)
table_df["Avg. Low Price"] = table_df.apply(
    lambda row: f"{row['Lowest Price Mean']:.2f}\n({row['Lowest Price SD']:.2f})",
    axis=1,
)

table_df = table_df[[
    "Algo. Rules",
    "N",
    r"$\Delta$",
    "Price Grid Min",
    "Price Grid Max",
    "Avg. Low Price",
]]
table_df[["Price Grid Min", "Price Grid Max"]] = table_df[["Price Grid Min", "Price Grid Max"]].round(2)

table_df

,Algo. Rules,N,$\Delta$,Price Grid Min,Price Grid Max,Avg. Low Price
0,3 Rules,3,0.98\n(0.04),1.28,2.09,1.99\n(0.02)
1,4 Rules,3,0.97\n(0.03),1.28,2.09,2.00\n(0.04)


In [3]:
table = (
    gt.GT(table_df)
    .cols_label(
        **{
            "Algo. Rules": "Algo. Rules",
            "N": "N",
            r"$\Delta$": r"$\Delta$",
            "Price Grid Min": r"$p_{min}$",
            "Price Grid Max": r"$p_{max}$",
            "Avg. Low Price": "Avg. Low Price",
        }
    )
    .fmt_number(columns=["Price Grid Min", "Price Grid Max"], decimals=2)
    .fmt_integer(columns=["N"])
)

table

Algo. Rules,N,$\Delta$,$p_{min}$,$p_{max}$,Avg. Low Price
3 Rules,3,0.98 (0.04),1.28,2.09,1.99 (0.02)
4 Rules,3,0.97 (0.03),1.28,2.09,2.00 (0.04)


In [4]:
output_path = OUTPUT_DIR / "exp03_k_action_mu0.25_pub.tex"

latex_str = table.as_latex()
latex_str = latex_str.replace(r'\$\\Delta\$', r'$\Delta$')
latex_str = latex_str.replace(r'\$p\_\{min\}\$', r'$p_{min}$')
latex_str = latex_str.replace(r'\$p\_\{max\}\$', r'$p_{max}$')
latex_str = latex_str.replace(
    "\n4 Rules",
    "\n\\midrule\\addlinespace[2.5pt]\n4 Rules",
)

latex_str = re.sub(r"\\begin\{table\}\[!t\]\s*", "", latex_str)
latex_str = re.sub(r"\\end\{table\}\s*", "", latex_str)
latex_str = re.sub(r"\\caption\*?\{[\s\S]*?\}\s*", "", latex_str, flags=re.S)
latex_str = re.sub(r"\n{3,}", "\n\n", latex_str).strip() + "\n"

output_path.write_text(latex_str)

print(f"Wrote LaTeX to: {output_path}")
print("\n--- LaTeX Preview ---\n")
print(latex_str)

Wrote LaTeX to: /Users/liushijian/Documents/GitHub/Amazon_BuyBox_Reinforcement_Learning/pricing_marl/analysis/robust_exp03_K_action/exp03_k_action_mu0.25_pub.tex

--- LaTeX Preview ---

\fontsize{12.0pt}{14.4pt}\selectfont

\begin{tabular*}{\linewidth}{@{\extracolsep{\fill}}lrlrrl}
\toprule
Algo. Rules & N & $\Delta$ & $p_{min}$ & $p_{max}$ & Avg. Low Price \\ 
\midrule\addlinespace[2.5pt]
3 Rules & 3 & 0.98
(0.04) & 1.28 & 2.09 & 1.99
(0.02) \\
\midrule\addlinespace[2.5pt]
4 Rules & 3 & 0.97
(0.03) & 1.28 & 2.09 & 2.00
(0.04) \\
\bottomrule
\end{tabular*}

